# Random_Forest — otimização de hiperparâmetros

**Lê:** `modeling_frame.csv` (só treino).

**Grava:** `outputs/results/best_params_Random_Forest.json` (e atualiza chave em `best_params.json`).

Não usa o conjunto de teste.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Image, Markdown
import pandas as pd

# sobe até achar analyses/grupo2
here = Path.cwd().resolve()
group = None
for c in [here, *here.parents]:
    if (c / 'config.py').exists() and (c / 'lib').is_dir():
        group = c
        break
if group is None:
    raise RuntimeError('Abra/rode este notebook de dentro de analyses/grupo2/...')
if str(group) not in sys.path:
    sys.path.insert(0, str(group))

import config
from lib import data as data_lib
from lib import features, eda_stl, models
from lib.nb_boot import require_file, save_json, load_json
data_lib.ensure_output_dirs()
print('grupo2 →', group)
print('outputs →', config.OUTPUT_DIR)


In [ ]:
model_df = pd.read_csv(require_file(config.DATA_DIR / 'modeling_frame.csv', '01_dados/03_feature_engineering.ipynb'),
                       parse_dates=[config.DATETIME_COL])
info = load_json(require_file(config.DATA_DIR / 'split_info.json', '01_dados/03_feature_engineering.ipynb'))
cut = int(info['cut_index'])
train = model_df.iloc[:cut].copy()
test = model_df.iloc[cut:].copy()
print('train', len(train), 'test', len(test))


In [ ]:
print('Otimizando Random_Forest...')
params = models.tune_random_forest(train)
save_json(params, config.RESULT_DIR / 'best_params_Random_Forest.json')
all_params = {}
bp = config.RESULT_DIR / 'best_params.json'
if bp.exists():
    all_params = load_json(bp)
all_params['Random_Forest'] = params
save_json(all_params, bp)
print('salvo')
display(params)